# Movie Database Encapsulator

## 1. Introduction

This notebook develops a Python database encapsulator for the Movie Database research project.

The encapsulator connects to the PostgreSQL database and provides reusable methods for retrieving research datasets as Pandas DataFrames.

By encapsulating SQL queries within a Python class, the database structure and query complexity are hidden from the user.

The encapsulator supports descriptive statistics, movie-level sales analysis, and the investigation of emotional characteristics in critic and user reviews.

In [ ]:
import pandas as pd
import psycopg2



Both libraries are working!


## 2. Database Connection

The encapsulator connects to the PostgreSQL database using psycopg2.

Database credentials are requested securely rather than stored directly in the notebook.

In [3]:
from getpass import getpass

db_config = {
    "host": "localhost",
    "port": 5432,
    "dbname": "movie_db_master",
    "user": "postgres",
    "password": getpass("Enter PostgreSQL password: ")
}

connection = psycopg2.connect(**db_config)

print("Successfully connected to the PostgreSQL database!")
connection.close()

Successfully connected to the PostgreSQL database!


## 3. Database Encapsulator

The MovieDatabase class encapsulates the PostgreSQL connection and SQL query execution.

It provides reusable methods that return query results as Pandas DataFrames, hiding the underlying database complexity from the user.

In [4]:
class MovieDatabase:

    def __init__(self, config):
        """Store the PostgreSQL database configuration."""
        self.config = config

    def execute_query(self, query):
        """Execute an SQL query and return a Pandas DataFrame."""

        with psycopg2.connect(**self.config) as connection:
            with connection.cursor() as cursor:
                cursor.execute(query)

                columns = [column[0] for column in cursor.description]
                rows = cursor.fetchall()

        return pd.DataFrame(rows, columns=columns)

    def get_review_score_summary(self):
        """
        Return descriptive statistics for critic and user review scores.
        User scores are multiplied by 10 to use the same scale as critic scores.
        """

        query = """
        SELECT
            'critic' AS reviewer_type,
            COUNT(*) AS n_reviews,
            ROUND(AVG(idvscore), 2) AS mean_score,
            ROUND(STDDEV(idvscore), 2) AS sd_score,
            MIN(idvscore) AS min_score,
            MAX(idvscore) AS max_score
        FROM critic_reviews
        WHERE idvscore IS NOT NULL

        UNION ALL

        SELECT
            'user',
            COUNT(*),
            ROUND(AVG(idvscore * 10), 2),
            ROUND(STDDEV(idvscore * 10), 2),
            MIN(idvscore * 10),
            MAX(idvscore * 10)
        FROM user_reviews
        WHERE idvscore IS NOT NULL;
        """

        return self.execute_query(query)

    def get_movie_sales_data(self):
        """
        Return movie scores, worldwide box-office revenue,
        production budget, runtime, and release date.
        """

        query = """
        SELECT
            m.movie_id,
            m.title,
            m.metascore,
            m.userscore,
            s.domestic_box_office + s.international_box_office
                AS worldwide_box_office,
            s.production_budget,
            s.runtime_minutes,
            s.release_date
        FROM movies m
        JOIN sales s
            ON s.movie_id = m.movie_id
        WHERE m.metascore IS NOT NULL
          AND m.userscore IS NOT NULL
          AND s.domestic_box_office IS NOT NULL
          AND s.international_box_office IS NOT NULL
        ORDER BY m.movie_id;
        """

        return self.execute_query(query)

### 3.1 Descriptive Statistics and Movie Sales Data

The first two methods provide datasets for the initial exploration of the movie database.

The `get_review_score_summary()` method compares critic and user review scores by calculating the number of reviews, mean score, standard deviation, minimum score, and maximum score. User review scores are multiplied by 10 to ensure comparability with critic scores.

The `get_movie_sales_data()` method joins the movies and sales tables using `movie_id`. It retrieves movie ratings, worldwide box-office revenue, production budget, runtime, and release date.

Both methods return Pandas DataFrames, allowing the research team to access and analyse the data without writing SQL queries directly.

In [5]:
# Create an instance of the database encapsulator
db = MovieDatabase(db_config)

# Test query 1: Review score statistics
print("REVIEW SCORE SUMMARY")
display(db.get_review_score_summary())

# Test query 2: Movie and sales information
print("MOVIE SALES DATA")
movie_sales = db.get_movie_sales_data()

display(movie_sales.head())
print("Number of movies:", len(movie_sales))

REVIEW SCORE SUMMARY


,reviewer_type,n_reviews,mean_score,sd_score,min_score,max_score
0,critic,234389,62.75,20.71,0,100
1,user,312826,65.86,31.87,0,100


MOVIE SALES DATA


,movie_id,title,metascore,userscore,worldwide_box_office,production_budget,runtime_minutes,release_date
0,2,10 Cloverfield Lane,76,7.7,108286422,15000000.0,103.0,2016-03-11
1,4,10 Years,61,6.9,987640,NaN,100.0,2012-09-14
2,13,10th & Wolf,36,6.7,143782,8000000.0,NaN,2006-08-18
3,20,12 Mighty Orphans,44,6.6,3712409,NaN,118.0,2021-06-11
4,22,12 Rounds,38,5.0,17306648,20000000.0,108.0,2009-03-27


Number of movies: 4890


In [6]:
class MovieDatabase:

    def __init__(self, config):
        """Store the PostgreSQL database configuration."""
        self.config = config

    def execute_query(self, query):
        """Execute a SQL query and return a Pandas DataFrame."""
        connection = psycopg2.connect(**self.config)

        try:
            with connection.cursor() as cursor:
                cursor.execute(query)
                columns = [column[0] for column in cursor.description]
                rows = cursor.fetchall()

            return pd.DataFrame(rows, columns=columns)

        finally:
            connection.close()

    def get_review_score_summary(self):
        """Return descriptive statistics for critic and user review scores."""
        query = """
        SELECT
            'critic' AS reviewer_type,
            COUNT(*) AS n_reviews,
            ROUND(AVG(idvscore), 2) AS mean_score,
            ROUND(STDDEV(idvscore), 2) AS sd_score,
            MIN(idvscore) AS min_score,
            MAX(idvscore) AS max_score
        FROM critic_reviews
        WHERE idvscore IS NOT NULL

        UNION ALL

        SELECT
            'user',
            COUNT(*),
            ROUND(AVG(idvscore * 10), 2),
            ROUND(STDDEV(idvscore * 10), 2),
            MIN(idvscore * 10),
            MAX(idvscore * 10)
        FROM user_reviews
        WHERE idvscore IS NOT NULL;
        """
        return self.execute_query(query)

    def get_movie_sales_data(self):
        """Return movie scores, revenue, and production details."""
        query = """
        SELECT
            m.movie_id,
            m.title,
            m.metascore,
            m.userscore,
            s.domestic_box_office + s.international_box_office
                AS worldwide_box_office,
            s.production_budget,
            s.runtime_minutes,
            s.release_date
        FROM movies m
        JOIN sales s
            ON s.movie_id = m.movie_id
        WHERE m.metascore IS NOT NULL
          AND m.userscore IS NOT NULL
          AND s.domestic_box_office IS NOT NULL
          AND s.international_box_office IS NOT NULL
        ORDER BY m.movie_id;
        """
        return self.execute_query(query)

    def get_critic_review_data(self):
        """Return critic reviews together with movie information for H2."""

        # JP: I join critic reviews with movies so I can see review emotions together with the Metascore.
        query = """
        SELECT
            cr.movie_id,
            m.title,
            m.metascore,
            cr.posemo,
            cr.negemo,
            cr.tone
        FROM critic_reviews AS cr
        JOIN movies AS m
            ON cr.movie_id = m.movie_id;
        """

        return self.execute_query(query)

    def get_h2_data(self):
        """Return critic review emotions, Metascore, and worldwide box office for H2."""

        # JP: I join critic reviews, movies and sales so I can compare review emotions with worldwide box office.
        query = """
        SELECT
            m.title,
            m.metascore,
            cr.posemo,
            cr.negemo,
            cr.tone,
            s.domestic_box_office + s.international_box_office AS worldwide_box_office
        FROM critic_reviews AS cr
        JOIN movies AS m
            ON cr.movie_id = m.movie_id
        JOIN sales AS s
            ON cr.movie_id = s.movie_id;
        """

        return self.execute_query(query)



### 3.2 H2 – Critic Review Emotions and Box-Office Performance

The first method joins critic reviews with movies so the review emotions can be viewed together with the Metascore.

The second method adds sales information and calculates worldwide box office. These two datasets provide the information needed to explore H2.

The encapsulator prepares the data but does not perform the statistical analysis itself.


In [7]:
# Recreate the encapsulator using the updated class
db = MovieDatabase(db_config)

# JP: I retrieve critic review and movie information for my H2 investigation.
critic_movie_data = db.get_critic_review_data()

# JP: I retrieve critic review, movie and sales information for my H2 investigation.
h2_data = db.get_h2_data()

# JP: I display the first five rows of the H2 data to check the results.
display(h2_data.head())


H2 – CRITIC REVIEW EMOTIONS AND BOX OFFICE


,movie_id,average_positive_emotion,average_negative_emotion,average_tone,metascore,worldwide_box_office
0,8435,2.9612500000000000,3.2533333333333333,41.9537500000000000,30,27147991
1,6717,4.8400000000000000,3.6152000000000000,42.8564000000000000,45,62761005
2,9876,4.5661538461538462,1.0679487179487179,63.4838461538461538,90,378985
3,8478,7.3104000000000000,2.3612000000000000,63.3072000000000000,82,716634
4,3172,4.8445714285714286,1.3608571428571429,64.3971428571428571,58,62989834


Number of movies: 5072
Number of columns: 6


In [8]:
class MovieDatabase:

    def __init__(self, config):
        """Store the PostgreSQL database configuration."""
        self.config = config

    def execute_query(self, query):
        """Execute a SQL query and return a Pandas DataFrame."""
        connection = psycopg2.connect(**self.config)

        try:
            with connection.cursor() as cursor:
                cursor.execute(query)
                columns = [column[0] for column in cursor.description]
                rows = cursor.fetchall()

            return pd.DataFrame(rows, columns=columns)

        finally:
            connection.close()

    def get_review_score_summary(self):
        """Return descriptive statistics for critic and user review scores."""
        query = """
        SELECT
            'critic' AS reviewer_type,
            COUNT(*) AS n_reviews,
            ROUND(AVG(idvscore), 2) AS mean_score,
            ROUND(STDDEV(idvscore), 2) AS sd_score,
            MIN(idvscore) AS min_score,
            MAX(idvscore) AS max_score
        FROM critic_reviews
        WHERE idvscore IS NOT NULL

        UNION ALL

        SELECT
            'user',
            COUNT(*),
            ROUND(AVG(idvscore * 10), 2),
            ROUND(STDDEV(idvscore * 10), 2),
            MIN(idvscore * 10),
            MAX(idvscore * 10)
        FROM user_reviews
        WHERE idvscore IS NOT NULL;
        """
        return self.execute_query(query)

    def get_movie_sales_data(self):
        """Return movie scores, revenue, and production details."""
        query = """
        SELECT
            m.movie_id,
            m.title,
            m.metascore,
            m.userscore,
            s.domestic_box_office + s.international_box_office
                AS worldwide_box_office,
            s.production_budget,
            s.runtime_minutes,
            s.release_date
        FROM movies m
        JOIN sales s
            ON s.movie_id = m.movie_id
        WHERE m.metascore IS NOT NULL
          AND m.userscore IS NOT NULL
          AND s.domestic_box_office IS NOT NULL
          AND s.international_box_office IS NOT NULL
        ORDER BY m.movie_id;
        """
        return self.execute_query(query)

    def get_critic_review_data(self):
        """Return critic reviews together with movie information for H2."""

        # JP: I join critic reviews with movies so I can see review emotions together with the Metascore.
        query = """
        SELECT
            cr.movie_id,
            m.title,
            m.metascore,
            cr.posemo,
            cr.negemo,
            cr.tone
        FROM critic_reviews AS cr
        JOIN movies AS m
            ON cr.movie_id = m.movie_id;
        """

        return self.execute_query(query)

    def get_h2_data(self):
        """Return critic review emotions, Metascore, and worldwide box office for H2."""

        # JP: I join critic reviews, movies and sales so I can compare review emotions with worldwide box office.
        query = """
        SELECT
            m.title,
            m.metascore,
            cr.posemo,
            cr.negemo,
            cr.tone,
            s.domestic_box_office + s.international_box_office AS worldwide_box_office
        FROM critic_reviews AS cr
        JOIN movies AS m
            ON cr.movie_id = m.movie_id
        JOIN sales AS s
            ON cr.movie_id = s.movie_id;
        """

        return self.execute_query(query)

    def get_h3_data(self):
        """
        Return movie-level user review emotions, Userscore,
        and post-release box-office trajectory for H3 analysis.

        The trajectory is calculated as domestic box office
        divided by opening weekend revenue.
        """
        query = """
        WITH user_emotions AS (
            SELECT
                movie_id,
                COUNT(*) AS number_user_reviews,
                AVG(posemo) AS avg_user_posemo,
                AVG(negemo) AS avg_user_negemo,
                AVG(tone) AS avg_user_tone
            FROM user_reviews
            GROUP BY movie_id
        )

        SELECT
            m.movie_id,
            m.title,
            m.userscore,
            s.opening_weekend,
            s.domestic_box_office,
            s.domestic_box_office::numeric
                / NULLIF(s.opening_weekend, 0)
                AS post_release_trajectory,
            u.number_user_reviews,
            u.avg_user_posemo,
            u.avg_user_negemo,
            u.avg_user_tone
        FROM movies AS m
        INNER JOIN sales AS s
            ON m.movie_id = s.movie_id
        INNER JOIN user_emotions AS u
            ON m.movie_id = u.movie_id
        WHERE m.userscore IS NOT NULL
          AND s.opening_weekend IS NOT NULL
          AND s.opening_weekend > 0
          AND s.domestic_box_office IS NOT NULL
        ORDER BY post_release_trajectory DESC;
        """
        return self.execute_query(query)

### 3.3 H3 – User Review Emotions and Post-Release Box-Office Trajectory

This method prepares the dataset for investigating whether the emotional characteristics of user reviews explain additional variation in a movie's post-release box-office trajectory beyond the Userscore alone.

Since multiple user reviews can belong to the same movie, positive emotion (posemo), negative emotion (negemo), and tone are aggregated by calculating their average values per movie.

Post-release box-office trajectory is calculated by dividing domestic box-office revenue by opening-weekend revenue.

Only movies with a valid Userscore, domestic box-office revenue, and a positive opening-weekend revenue are included.

The resulting dataset contains one observation per movie and can be used for further statistical analysis to investigate H3.

In [9]:
# Recreate the encapsulator using the updated class
db = MovieDatabase(db_config)

# Retrieve the H3 dataset
h3_data = db.get_h3_data()

# Display the first five movies
print("H3 – USER REVIEW EMOTIONS AND BOX-OFFICE TRAJECTORY")
display(h3_data.head())

# Check the dataset
print("Number of movies:", len(h3_data))
print("Number of columns:", len(h3_data.columns))

H3 – USER REVIEW EMOTIONS AND BOX-OFFICE TRAJECTORY


,movie_id,title,userscore,opening_weekend,domestic_box_office,post_release_trajectory,number_user_reviews,avg_user_posemo,avg_user_negemo,avg_user_tone
0,3292,Frozen,7.5,243390,400953009,1647.3684580303217059,356,7.3929494382022472,2.1361516853932584,73.3612921348314607
1,6383,Paranormal Activity,5.9,77873,107918810,1385.8309041644729239,322,3.7034472049689441,6.6244720496894410,22.5847515527950311
2,5111,Lone Survivor,7.5,90872,125095601,1376.6132692138392464,92,5.4428260869565217,3.6738043478260870,52.3520652173913043
3,3953,Hostiles,7.2,22849,29819114,1305.0511619764541118,60,5.6623333333333333,2.7588333333333333,57.0591666666666667
4,7557,Snatch.,8.7,27932,30093107,1077.3702921380495489,59,7.8525423728813559,1.5332203389830508,74.9625423728813559


Number of movies: 5832
Number of columns: 10


### 3.4 Data Quality Assessment

Before conducting statistical analysis, the H3 dataset is checked for completeness and consistency.

The validation examines the total number of movies, the number of unique movie IDs, and the presence of duplicate records.

Missing values are checked for all variables to identify potential data quality issues.

Finally, the output is verified to ensure that the encapsulator returns a Pandas DataFrame.

These checks help assess whether the retrieved dataset is suitable for further statistical analysis.

In [10]:
print("H3 DATA QUALITY CHECK")

print("Total movies:", len(h3_data))
print("Unique movie IDs:", h3_data["movie_id"].nunique())
print("Duplicate movie IDs:", h3_data["movie_id"].duplicated().sum())

print("\nMissing values per column:")
display(h3_data.isnull().sum())

print("\nIs this a Pandas DataFrame?")
print(isinstance(h3_data, pd.DataFrame))

H3 DATA QUALITY CHECK
Total movies: 5832
Unique movie IDs: 5832
Duplicate movie IDs: 0

Missing values per column:


movie_id                   0
title                      0
userscore                  0
opening_weekend            0
domestic_box_office        0
post_release_trajectory    0
number_user_reviews        0
avg_user_posemo            0
avg_user_negemo            0
avg_user_tone              0
dtype: int64


Is this a Pandas DataFrame?
True


## 4. Validation and Data Quality

The database encapsulator was developed to provide a reusable Python interface for accessing the PostgreSQL movie database.

Five methods are implemented to retrieve descriptive statistics, movie-level sales information, critic review and movie information (H2), critic review and sales information (H2), and user review emotions (H3).

### Validation Results

The methods are designed to return Pandas DataFrames for further statistical analysis.

Additional data quality checks are performed on the H3 dataset. All 5,832 movie IDs were unique, no duplicate movie IDs were identified, and no missing values were found in the returned variables.

### Encapsulation

The `MovieDatabase` class hides the technical complexity of database connections, SQL execution, and data retrieval. Each research dataset can be accessed through a dedicated method that returns a Pandas DataFrame.

This approach improves code reusability, consistency, and accessibility for further statistical analysis.

### Limitations

The datasets include only movies meeting the completeness requirements specified in their respective SQL queries. Consequently, the number of observations differs between methods.

Furthermore, extremely high post-release trajectory ratios may occur when a movie has a relatively small opening-weekend revenue. These observations should be investigated during statistical analysis rather than automatically removed.


## 5. Instructions for Group Members

### Requirements

To use this database encapsulator, the following are required:

- Python 3
- PostgreSQL
- pandas
- psycopg2
- Jupyter Notebook or Visual Studio Code

### Database Access

The encapsulator connects to the PostgreSQL database `movie_db_master`.

Each group member must have access to a PostgreSQL database containing the same tables and data.

If necessary, the database configuration (host, port, database name, and username) can be adjusted.

The PostgreSQL password is entered securely using `getpass()` and is not stored directly in the notebook.

### How to Use

1. Open `database_encapsulator.ipynb`.
2. Select the correct Python kernel.
3. Run all notebook cells.
4. Enter the PostgreSQL password when requested.
5. Use the MovieDatabase methods to retrieve the required datasets.

### Available Methods

- `db.get_review_score_summary()` – Descriptive statistics for critic and user review scores.
- `db.get_movie_sales_data()` – Movie scores, worldwide box office, and production information.
- `db.get_critic_review_data()` – Critic reviews, movie titles, and Metascores for JP's H2 investigation.
- `db.get_h2_data()` – Critic review emotions, Metascore, and worldwide box office for JP's H2 investigation.
- `db.get_h3_data()` – User review emotions, Userscore, and post-release box-office trajectory.

All methods return Pandas DataFrames that can be used for further statistical analysis.
